In [67]:
import numpy as np
import pandas as pd
import re
import gender_guesser.detector as gender

# Functions
def has_digits(name: str) -> bool:
    ''' 
    Returns 'True' if name contains digits, else 'False'
    '''
    if not isinstance(name, str):
        return False
    return any(c.isdigit() for c in name)

d = gender.Detector()
def infer_gender(name):
    if pd.isna(name):
        return np.nan
    first = str(name).split()[0]
    g = d.get_gender(first)
    return 'M' if g in ['male', 'mostly_male'] else 'F' if g in ['female', 'mostly_female'] else np.nan

df=pd.read_csv('employee_records[2].csv')

# FullName
df['FullName']=df['FullName'].apply(lambda x: np.nan if has_digits(x) else x)
df['FullName']=df['FullName'].str.title()

# Age
df['Age']=df['Age'].fillna(df['Age'].median())
df['Age']=df['Age'].round(0).astype(int)

# Gender
df['Gender']=df['Gender'].str.upper()
df['Gender']=df['Gender'].map({'MALE': 'M', 'FEMALE': 'F'})
df['Gender'] = df['Gender'].fillna(df['FullName'].apply(infer_gender))

# EmailAddress
email_pattern = r"^(?!.*\.\.)[\w\.-]+@[\w-]+(\.[\w-]+)+$"
df['EmailAddress']=df['EmailAddress'].apply(lambda x: x if re.match(email_pattern, str(x)) else np.nan)
df['EmailMissing'] = df['EmailAddress'].isnull()

# Salary
df['Salary']=pd.to_numeric(df['Salary'], errors='coerce')
df['Salary']=df['Salary'].fillna(df['Salary'].median())

# StartDate
def try_parse_date(date_str):
    if pd.isna(date_str):
        return pd.NaT
    
    # Remove spaces
    date_str = str(date_str).strip()

    # ISO format: YYYY-MM-DD or YYYY/MM/DD
    iso_pattern = r"^\d{4}[-/]\d{2}[-/]\d{2}$"

    try:
        if re.match(iso_pattern, date_str):
            # Parse normally (ISO)
            return pd.to_datetime(date_str, errors='coerce')
        else:
            # Try day-first formats (like 15-02-2020)
            return pd.to_datetime(date_str, errors='coerce', dayfirst=True)
    except:
        return pd.NaT

df['StartDate']=df['StartDate'].apply(try_parse_date)
df['StartDate'] = df['StartDate'].fillna(df['StartDate'].median())

# Department
df['Department']=df['Department'].str.replace('.', '')
df['Department']=df['Department'].str.strip().str.title()
department_map = {
    'Eng': 'Engineering',
    'Hr': 'Human Resources',
    'Prdct': 'Product',
    'Mkt': 'Marketing', 'Mktg': 'Marketing',
    'Ops': 'Operations'
}
df['Department']=df['Department'].map(department_map)
df=pd.get_dummies(df, columns=['Department'], drop_first=True)

# Others
df=df.drop_duplicates()

# Output
# Force Display
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
df
# df.isna().sum()
# for col in df.columns:
#     print(f"\n{col} ➤ Unique values:")
#     print(df[col].dropna().unique())

,FullName,Age,Gender,EmailAddress,Salary,StartDate,EmailMissing,Department_Human Resources,Department_Marketing,Department_Operations,Department_Product
0,Norma Fisher,38,F,ysullivan@yahoo.com,95000.00,2025-06-07,False,False,False,False,False
1,Colleen Taylor,28,F,montgomeryjohn@mcgrath.com,95000.00,2021-02-14,False,False,False,False,False
2,Brian Hamilton,50,M,hramos@brown-sellers.com,95000.00,2020-04-26,False,False,False,False,False
3,NaN,34,NaN,leeashley@gmail.com,95000.00,2017-12-06,False,False,False,False,False
4,Willie Golden,51,M,turnerkelly@gmail.com,95000.00,2020-04-26,False,True,False,False,False
5,Robert Payne,48,M,zdavis@yahoo.com,95000.00,2019-08-17,False,True,False,False,False
6,Collin Lopez,38,F,NaN,95000.00,2018-01-19,True,False,False,False,False
7,Tammy Fernandez,38,F,aimee33@hotmail.com,95000.00,2020-04-26,False,True,False,False,False
8,Matthew Davies,42,M,nguyendarrell@hotmail.com,95000.00,2018-09-29,False,True,False,False,False
9,NaN,37,NaN,marvincabrera@gmail.com,95000.00,2020-04-26,False,False,False,False,False
